In [1]:
# Import libraries
import numpy as np
import pandas as pd

In [2]:
# Read CSV File
df = pd.read_csv("d:/Dataset/accepted_2007_to_2018Q4.csv", low_memory=False)

In [3]:
# Keep only the variables we will use
df = df[[
    'loan_status',
    'int_rate',
    'term',
    'fico_range_low',
    'dti',
    'avg_cur_bal',
    'mort_acc',
    'loan_amnt',
    'inq_last_6mths',
    'bc_util',
    'revol_util',
    'grade',
    'verification_status',
    'home_ownership',
    'purpose']].copy()

# Remove rows where 'loan_status' is NA
df = df[df["loan_status"].notna() & (df["loan_status"].str.strip() != "")]
# Loan Status
df['loan_status'] = df['loan_status'].str.replace(
    "Does not meet the credit policy. Status:Charged Off",
    "Charged Off"
)
df['loan_status'] = df['loan_status'].str.replace(
    "Does not meet the credit policy. Status:Fully Paid",
    "Fully Paid"
)
# Keep only rows where loan_status = "Charged Off" or "Fully Paid"
df = df[df['loan_status'].isin(['Charged Off', 'Fully Paid'])].copy()

# Verification Status
# Combine 'Source Verified' and 'Verified'
df["verification_status"] = df["verification_status"].str.replace(
    "Source Verified", 
    "Verified"
)
df["verification_status"] = df["verification_status"].str.replace(
    "Not Verified", 
    "Not_Verified"
)
df['term'] = df['term'].str.replace(
    " months",
    ""
)
df['term'] = df['term'].str.replace(
    " ",
    ""
)

# Ran into some nulls that will need to be addressed before Regression
# Columns where NaN should be replaced with the median within each grade
cols_to_impute = ['avg_cur_bal', 'mort_acc', 'bc_util', 'dti', 'inq_last_6mths', 'revol_util']

# Replace NaN with the median value within each grade
for col in cols_to_impute:
    df[col] = df.groupby('grade')[col].transform(
        lambda x: x.fillna(x.median())
    )

# Create dummy variables for all categorical variables
# loan_status, term, verification_status, home_ownership, purpose
columns_to_encode = ['loan_status','term','verification_status','home_ownership','purpose']
df_dummies = pd.get_dummies(
    df,
    columns= columns_to_encode,
    drop_first = True,
    dtype = int
)
df_dummies['loan_status_Charged Off'] = 1 - df_dummies['loan_status_Fully Paid']

In [4]:
df_dummies.head()

,int_rate,fico_range_low,dti,avg_cur_bal,mort_acc,loan_amnt,inq_last_6mths,bc_util,revol_util,grade,...,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,loan_status_Charged Off
0,13.99,675.0,5.91,20701.0,1.0,3600.0,1.0,37.2,29.7,C,...,0,0,0,0,0,0,0,0,0,0
1,11.99,715.0,16.06,9733.0,4.0,24700.0,4.0,27.1,19.2,C,...,0,0,0,0,0,0,1,0,0,0
2,10.78,695.0,10.78,31617.0,5.0,20000.0,0.0,55.9,56.2,B,...,0,0,0,0,0,0,0,0,0,0
4,22.45,695.0,25.37,27644.0,6.0,10400.0,3.0,77.5,64.5,F,...,0,1,0,0,0,0,0,0,0,0
5,13.44,690.0,10.20,2560.0,0.0,11950.0,0.0,91.0,68.4,C,...,0,0,0,0,0,0,0,0,0,0


In [5]:
print("Original variable names are:")
for col in df.columns:
    print(col)
print("\n\nDummy variable names:")    
for col in df_dummies.columns:
    print(col)

Original variable names are:
loan_status
int_rate
term
fico_range_low
dti
avg_cur_bal
mort_acc
loan_amnt
inq_last_6mths
bc_util
revol_util
grade
verification_status
home_ownership
purpose


Dummy variable names:
int_rate
fico_range_low
dti
avg_cur_bal
mort_acc
loan_amnt
inq_last_6mths
bc_util
revol_util
grade
loan_status_Fully Paid
term_60
verification_status_Verified
home_ownership_MORTGAGE
home_ownership_NONE
home_ownership_OTHER
home_ownership_OWN
home_ownership_RENT
purpose_credit_card
purpose_debt_consolidation
purpose_educational
purpose_home_improvement
purpose_house
purpose_major_purchase
purpose_medical
purpose_moving
purpose_other
purpose_renewable_energy
purpose_small_business
purpose_vacation
purpose_wedding
loan_status_Charged Off


In [6]:
# Can now do analysis on df_dummies

In [7]:
df.to_csv("LC_Cleaned_Data.csv", index = False)
df_dummies.to_csv("LC_Cleaned_Data_dummies.csv", index = False)